<a href="https://colab.research.google.com/github/gouravbembi/Ip-assignments/blob/main/Assignment11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import tensorflow as tf
from scipy.signal import correlate2d

matrix = np.random.randint(0, 10, (3, 3)).astype(np.float32)

print("Original Matrix:\n", matrix)

kernel = np.random.randint(0, 3, (3, 3)).astype(np.float32)

padded = np.pad(matrix, ((1, 1), (1, 1)), mode='constant')

convolution = correlate2d(padded, kernel, mode='valid')

print("\nKernel:\n", kernel)
print("\nPadded Matrix:\n", padded)
print("\nConvolution:\n", convolution)

relu = np.maximum(convolution, 0)
sigmoid = 1 / (1 + np.exp(-convolution))

print("\nReLU:\n", relu)
print("\nSigmoid:\n", sigmoid)

max_pool = np.max(matrix.reshape(1, 3, 3, 1), axis=(1, 2))
min_pool = np.min(matrix.reshape(1, 3, 3, 1), axis=(1, 2))
global_avg_pool = np.mean(matrix)

print("\nMax Pooling:", max_pool)
print("\nMin Pooling:", min_pool)
print("\nGlobal Average Pooling:", global_avg_pool)

flatten = matrix.flatten()

print("\nFlatten:\n", flatten)

dense = tf.keras.layers.Dense(4, activation='relu')
dense_output = dense(tf.constant(flatten.reshape(1, -1)))

print("\nDense Layer Output:\n", dense_output.numpy())

bn = tf.keras.layers.BatchNormalization()
bn_output = bn(tf.constant(matrix.reshape(1, 3, 3, 1)), training=True)

print("\nBatch Normalization:\n", bn_output.numpy().reshape(3, 3))

dropout = tf.keras.layers.Dropout(0.5)
dropout_output = dropout(tf.constant(flatten.reshape(1, -1)), training=True)

print("\nDropout Output:\n", dropout_output.numpy())

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("\nEarly Stopping Callback Created Successfully")
print(early_stopping)




Original Matrix:
 [[3. 6. 8.]
 [0. 7. 8.]
 [4. 9. 4.]]

Kernel:
 [[2. 0. 0.]
 [1. 2. 1.]
 [1. 0. 2.]]

Padded Matrix:
 [[0. 0. 0. 0. 0.]
 [0. 3. 6. 8. 0.]
 [0. 0. 7. 8. 0.]
 [0. 4. 9. 4. 0.]
 [0. 0. 0. 0. 0.]]

Convolution:
 [[26. 39. 29.]
 [25. 40. 44.]
 [17. 26. 31.]]

ReLU:
 [[26. 39. 29.]
 [25. 40. 44.]
 [17. 26. 31.]]

Sigmoid:
 [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]

Max Pooling: [[9.]]

Min Pooling: [[0.]]

Global Average Pooling: 5.4444447

Flatten:
 [3. 6. 8. 0. 7. 8. 4. 9. 4.]

Dense Layer Output:
 [[0.        2.0435202 0.        1.8707621]]

Batch Normalization:
 [[-0.88778925  0.20177019  0.92814314]
 [-1.9773487   0.5649568   0.92814314]
 [-0.5246028   1.2913295  -0.5246028 ]]

Dropout Output:
 [[ 6. 12. 16.  0. 14. 16.  0.  0.  0.]]

Early Stopping Callback Created Successfully
Dataset Shape: (600, 64, 64, 3)
Labels Shape: (600,)
Selected Classes: ['n01443537', 'n01629819', 'n01641577']
n01443537 : 200 images
n01629819 : 200 images
n01641577 : 200 images

Training Images: 4

In [ ]:

import os
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
zip_path = "/content/tiny-imagenet-200.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

extract_path = "/content"

if not os.path.exists("/content/tiny-imagenet-200"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

data_path = "/content/tiny-imagenet-200/train"

class_names = sorted(os.listdir(data_path))[:3]

X = []
y = []

for label, class_name in enumerate(class_names):

    image_folder = os.path.join(data_path, class_name, "images")

    image_files = sorted(os.listdir(image_folder))[:200]

    for image_file in image_files:

        image_path = os.path.join(image_folder, image_file)

        image = Image.open(image_path).convert("RGB")
        image = image.resize((64, 64))

        X.append(np.array(image))
        y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("Dataset Shape:", X.shape)
print("Labels Shape:", y.shape)
print("Selected Classes:", class_names)

for i, name in enumerate(class_names):
    print(name, ":", np.sum(y == i), "images")

X = X / 255.0

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\nTraining Images:", X_train.shape[0])
print("Validation Images:", X_val.shape[0])
print("Testing Images:", X_test.shape[0])


plt.figure(figsize=(10, 6))

for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X[i])
    plt.title(class_names[y[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = 3

custom_model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),

    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

custom_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

custom_model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = custom_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = np.argmax(custom_model.predict(X_test), axis=1)

print("Custom CNN Results")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro', zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average='macro', zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average='macro', zero_division=0))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))


In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, MobileNetV2, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def build_transfer_model(model_name):

    inputs = layers.Input(shape=(64, 64, 3))

    if model_name == "VGG16":
        x = layers.Lambda(lambda z: vgg_preprocess(z * 255.0))(inputs)
        base = VGG16(
            weights='imagenet',
            include_top=False,
            input_shape=(64, 64, 3)
        )

    elif model_name == "MobileNetV2":
        x = layers.Lambda(lambda z: mobile_preprocess(z * 255.0))(inputs)
        base = MobileNetV2(
            weights='imagenet',
            include_top=False,
            input_shape=(64, 64, 3)
        )

    elif model_name == "ResNet50":
        x = layers.Lambda(lambda z: resnet_preprocess(z * 255.0))(inputs)
        base = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(64, 64, 3)
        )

    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(3, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

models_dict = {
    "VGG16": build_transfer_model("VGG16"),
    "MobileNetV2": build_transfer_model("MobileNetV2"),
    "ResNet50": build_transfer_model("ResNet50")
}


transfer_histories = {}
transfer_results = {}

for name, model in models_dict.items():

    print("\nTraining:", name)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=16,
        callbacks=[early_stop]
    )

    transfer_histories[name] = history

    test_loss, test_accuracy = model.evaluate(
        X_test, y_test, verbose=0
    )

    transfer_results[name] = test_accuracy

    print(name, "Test Accuracy:", test_accuracy)


import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

results = {}

all_models = {
    "Custom CNN": custom_model,
    "VGG16": models_dict["VGG16"],
    "MobileNetV2": models_dict["MobileNetV2"],
    "ResNet50": models_dict["ResNet50"]
}

for name, model in all_models.items():

    predictions = model.predict(X_test, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, average='macro', zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, average='macro', zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, y_pred, average='macro', zero_division=0
        )
    }

results_df = pd.DataFrame(results).T

print("\nMODEL PERFORMANCE COMPARISON\n")
print(results_df)

results_df.plot(
    kind='bar',
    figsize=(12, 6),
    ylim=(0, 1),
    width=0.8
)

plt.title("Custom CNN vs Transfer Learning Models")
plt.ylabel("Score")
plt.xlabel("Models")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()
